In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_tipo_predio")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_tipo_predio (
  tipo_predio_id INT,
  tipo_predio_descripcion STRING
)
""")

In [0]:
import pandas as pd

df = spark.table("workspace.silver.predios_registro").toPandas()

In [0]:

tipo_predio = (
    df.reindex(
        columns=[
            "tipo_predio_id",
            "tipo_predio_descripcion",
        ]
    )
)

dim = (
    pd.concat([tipo_predio], ignore_index=True)
    .dropna(subset=["tipo_predio_id"])
    .drop_duplicates(subset=["tipo_predio_descripcion"])
    .sort_values("tipo_predio_descripcion", ignore_index=True)
)

df_spark = spark.createDataFrame(dim)



In [0]:

# Usamos overwrite para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_tipo_predio")